# DSA5208 Assignment 1

### Group Members

- Ashley Lee Ruoxin
- Koh Si Xing
- Kevin Royce Thomson

## 1. Project Setup

### 1.1 Purpose of This Notebook

This notebook investigates how MongoDB replica-set consistency and availability behave during database-node failures.

#### Configurations Compared

| Configuration | Write Concern | Read Concern | Expected Failover Behaviour |
|---|---|---|---|
| `majority` | `w="majority"` | `readConcern="majority"` | A write is confirmed only after reaching a majority, so an acknowledged write should survive primary failure. |
| `w1-local` | `w=1` | `readConcern="local"` | A write is confirmed once the primary stores it, so it may disappear if the primary fails before replication. |

#### Failure Scenarios

| Scenario | What Is Tested |
|---|---|
| One secondary unavailable | Whether writes and reads continue while the remaining two nodes still form a majority. |
| Two secondaries unavailable | Whether the primary loses its majority and how each write concern behaves without quorum. |
| Primary failure and failover | Whether a write survives after the primary fails, a new primary is elected, and the old primary rejoins. |

The experiments examine whether writes are acknowledged, remain visible, and survive recovery; how reads behave during failures; and whether client-centric consistency guarantees are preserved after failover.

### 1.2 Database Connections

Two types of MongoDB connection are used:

- `client` connects to the complete `dsa5208` replica set. It automatically sends writes to the current primary, including after failover when a different node is elected as primary.

- `direct` contains one direct connection for each node: `mongo1`, `mongo2`, and `mongo3`. These connections are used to inspect the local data stored on a specific node and to control replication failpoints during failure experiments.

In [1]:
from pymongo import MongoClient, ReadPreference
from pymongo.errors import ExecutionTimeout, PyMongoError, WTimeoutError, ServerSelectionTimeoutError
from pymongo.read_concern import ReadConcern
from pymongo.write_concern import WriteConcern

import subprocess
import time

In [2]:
REPLICA_SET_URI = "mongodb://mongo1:27017,mongo2:27017,mongo3:27017/?replicaSet=dsa5208"
DB_NAME = "university"
NODES = ["mongo1", "mongo2", "mongo3"]

client = MongoClient(REPLICA_SET_URI, serverSelectionTimeoutMS=5000)

direct = {
    node: MongoClient(f"mongodb://{node}:27017/?directConnection=true",
                      serverSelectionTimeoutMS=3000)
    for node in NODES
}

### 1.3 Consistency Configurations

The failure experiments compare two configurations with different write and read guarantees.

| Configuration | Write Concern | Read Concern | Meaning |
|---|---|---|---|
| `majority` | `w="majority"` | `readConcern="majority"` | A write is acknowledged only after reaching a majority of nodes. Reads return majority-committed data. |
| `w1-local` | `w=1` | `readConcern="local"` | A write is acknowledged once the primary stores it. Reads return the value currently stored on the selected node. |

The following timing values prevent operations from waiting indefinitely:

| Constant | Value | Purpose |
|---|---:|---|
| `READ_TIMEOUT_MS` | 3000 ms | Maximum time a read can wait before returning `TIMEOUT`. |
| `WTIMEOUT_MS` | 3000 ms | Maximum time a write can wait for its write concern to be satisfied. |
| `CATCHUP_SECONDS` | 2 seconds | Waiting period after nodes reconnect, allowing replication to catch up. |

The `majority` configuration represents the safer case: MongoDB does not confirm a write unless another node has also stored it. The `w1-local` configuration represents the weaker case: the primary can confirm a write before any secondary receives it, and reads may return stale local data from a lagging node.

In [3]:
CONFIGS = {
    "majority": {"write_concern": "majority", "read_concern": "majority"},
    "w1-local": {"write_concern": 1, "read_concern": "local"}
}

READ_TIMEOUT_MS = 3000   # how long a read may wait
WTIMEOUT_MS = 3000       # how long a write waits for confirmation from other nodes
CATCHUP_SECONDS = 2      # safety margin for nodes to catch up after reconnecting

## 2. Experiment Support

### 2.1 Collections and Test Data

The helper functions in this section create MongoDB collection handles and prepare consistent baseline data for each failure experiment.

- `get_collection()` returns a collection configured with the selected write concern, read concern, and read preference. The `target` parameter determines whether reads are directed to the current primary or to an available secondary.

- `reset_document()` creates or replaces a test document before an experiment begins. It uses `w=3`, meaning that all three replica-set nodes must store the baseline document before the operation is confirmed. This ensures that every node starts from the same known state before a node is paused, disconnected, or killed.

- `make_student()` creates a standard student test document. Optional keyword arguments can override default fields when an experiment requires different values.

- `count_rollback_lines()` counts `ROLLBACK` entries in a node’s Docker logs. It is used after a primary failure to check whether the old primary rolled back an unreplicated write before rejoining the replica set.

In [4]:
def get_collection(collection_name, config_name, target="primary"):
    """Return a collection handle with the config's write and read concern applied."""
    config = CONFIGS[config_name]
    read_preference = {
        "primary": ReadPreference.PRIMARY,
        "secondary": ReadPreference.SECONDARY,
    }[target]

    db = client.get_database(
        DB_NAME,
        write_concern=WriteConcern(w=config["write_concern"], wtimeout=WTIMEOUT_MS),
        read_concern=ReadConcern(config["read_concern"]),
        read_preference=read_preference,
    )
    return db[collection_name]


# Baseline writes wait for ALL three nodes, so every node starts from the same state.
SETUP_WRITE_CONCERN = WriteConcern(w=3, wtimeout=30000)


def reset_document(collection_name, id_field, document):
    """Create or overwrite a test document and wait until every node has it."""
    db = client.get_database(DB_NAME, write_concern=SETUP_WRITE_CONCERN)
    db[collection_name].replace_one({id_field: document[id_field]}, document, upsert=True)


def make_student(student_id, name, **overrides):
    student = {
        "student_id": student_id,
        "student_name": name,
        "student_status": "Undergraduate",
        "seniority": "Year 1",
        "academic_load": "Full Time",
    }
    student.update(overrides)
    return student

def count_rollback_lines(node):
    """Count ROLLBACK entries in a node's Docker log."""
    logs = subprocess.run(["docker", "logs", node], capture_output=True, text=True)
    return (logs.stdout + logs.stderr).count("ROLLBACK")

### 2.2 Reading, Writing, and Display Functions

The following helper functions perform database operations and present the results of each failure experiment.

- `read_field()` reads a specified field through the replica-set connection. It uses the selected configuration’s read concern and returns `TIMEOUT`, `NOT FOUND`, or an error message when the read cannot complete normally.

- `read_node()` reads a field directly from one specified node, such as `mongo1` or `mongo3`. This makes it possible to compare the local data stored on individual replicas during a failure or recovery.

- `try_write()` performs an update and reports whether MongoDB satisfied the selected write concern. For example, a majority write may return `NOT confirmed` when too many nodes are unavailable.

- `show()` prints labels and values in a consistent, readable format.

- `header()` displays the experiment name together with the active write concern and read concern.

In [5]:
def read_field(collection, query, field):
    """Read one field through the replica-set client, using the collection's read concern."""
    try:
        doc = collection.find_one(query, max_time_ms=READ_TIMEOUT_MS)
    except ExecutionTimeout:
        return "TIMEOUT"
    except PyMongoError as error:
        return f"ERROR ({type(error).__name__})"
    return doc.get(field) if doc else "NOT FOUND"


def read_node(node, query, field):
    """Read one field from a single node's own copy of the data."""
    try:
        doc = direct[node][DB_NAME]["students"].find_one(query)
    except PyMongoError as error:
        return f"ERROR ({type(error).__name__})"
    return doc.get(field) if doc else "NOT FOUND"


def try_write(collection, query, update):
    """Run an update and report whether the write concern was satisfied."""
    try:
        collection.update_one(query, update)
        return "acknowledged"
    except WTimeoutError:
        return f"NOT confirmed within {WTIMEOUT_MS} ms"


def show(label, value):
    print(f"  {label:<30} {value}")


def header(title, config_name):
    config = CONFIGS[config_name]
    print(f"=== {title} | w={config['write_concern']}, "
          f"readConcern={config['read_concern']} ===")

### 2.3 Node-Control Functions

#### Disconnecting and Reconnecting Replication

These functions control individual Docker containers and replica-set replication during failure experiments.

- `docker()` runs Docker commands to pause, unpause, kill, or start a specified MongoDB node.

- `disconnect_replication()` stops a node from receiving new replicated operations while keeping the node running and reachable.

- `reconnect_replication()` restores replication so that the disconnected node can catch up with the current primary.

- `current_primary()` identifies the current primary node. It is used after a primary failure to wait for and identify the newly elected primary.

- `wait_until_secondary()` waits until a restarted node has rejoined the replica set as a secondary, including after any rollback of unreplicated writes.

In [6]:
def docker(*args):
    """Run a docker command from the notebook, e.g. docker("kill", "mongo1")."""
    subprocess.run(["docker", *args], check=True, capture_output=True)
    print(f"  [docker {' '.join(args)}]")


def disconnect_replication(node):
    direct[node].admin.command({"configureFailPoint": "stopReplProducer", "mode": "alwaysOn"})
    time.sleep(2)


def reconnect_replication(node):
    direct[node].admin.command({"configureFailPoint": "stopReplProducer", "mode": "off"})


def current_primary(timeout=60):
    """Wait until the replica set has a primary and return its name, e.g. "mongo1"."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            return client.admin.command("hello")["primary"].split(":")[0]
        except (PyMongoError, KeyError):
            time.sleep(1)
    raise RuntimeError("No primary was elected in time")


def wait_until_secondary(node, timeout=120):
    """Wait until a restarted node has rejoined as a secondary (after any rollback)."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            if direct[node].admin.command("hello").get("secondary"):
                return
        except PyMongoError:
            pass
        time.sleep(2)
    raise RuntimeError(f"{node} did not rejoin as a secondary in time")

### 2.4 Cluster Status and Recovery

These functions inspect and restore the replica set after a failure experiment.

- `show_members()` displays each node’s current role, such as `PRIMARY` or `SECONDARY`, together with its latest applied operation time.

- `restore_cluster()` returns the cluster to a usable state if an experiment was interrupted or completed. It unpauses and starts all containers, restores replication on each node, waits for a primary to be elected, and displays the final replica-set status.

In [7]:
def show_members():
    """Print each member's state and whether it has applied everything the primary has."""
    members = client.admin.command("replSetGetStatus")["members"]
    primary = next((m for m in members if m["stateStr"] == "PRIMARY"), None)
  
    for m in members:
        print(m["name"], m["stateStr"], m.get("optimeDate"))


def restore_cluster():
    """
    Undo anything these tests change, in case a run was interrupted:
    unpause and start every container, then turn replication back on.
    Errors are ignored because most nodes will already be in the right state.
    """
    for node in NODES:
        subprocess.run(["docker", "unpause", node], capture_output=True)
        subprocess.run(["docker", "start", node], capture_output=True)
    for node in NODES:
        try:
            reconnect_replication(node)
        except PyMongoError:
            pass  # a freshly restarted node has no failpoints set anyway
    print("Primary:", current_primary())
    show_members()

### 2.5 Secondary Failure Simulation Function

The `run_disconnected_secondaries()` function simulates one or two secondary nodes becoming unavailable.

`docker pause` freezes the selected container without deleting its data or changing the Docker network configuration. To the replica set and the client, the paused node stops responding: it cannot serve reads, receive replication, or participate in elections.

The function:

1. Identifies the current primary and selects one or two other nodes to pause.
2. Resets a configuration-specific student document on all three nodes.
3. Pauses the selected secondary nodes.
4. Attempts to update the student document through the primary.
5. Records whether the selected write concern was satisfied and what the primary read returns.
6. Unpauses the disconnected nodes.
7. Waits for replication catch-up and displays the value stored on every node.

In [8]:
def run_disconnected_secondaries(config_name, how_many=1):
    header(f"Disconnected secondaries: {how_many}", config_name)
    students_primary = get_collection("students", config_name, "primary")

    primary = current_primary()
    paused = [node for node in NODES if node != primary][:how_many]

    student_id = f"S006-{config_name}-{how_many}"
    query = {"student_id": student_id}
    reset_document("students", "student_id", make_student(student_id, "Robert Tan"))

    for node in paused:
        docker("pause", node)
    try:
        update = {"$set": {"student_status": "Graduate"}}
        show("Write result:", try_write(students_primary, query, update))
        show("Primary read:", read_field(students_primary, query, "student_status"))
    except ServerSelectionTimeoutError as error:
        print("Experiment stopped")
    finally:
        for node in paused:
            docker("unpause", node)

    time.sleep(CATCHUP_SECONDS)
    for node in NODES:
        show(f"{node} after reconnecting:", read_node(node, query, "student_status"))

### 2.6 Primary Failure and Failover Simulation Function

The `run_downed_primary()` function simulates a primary failure immediately after a write.

1. Both secondaries are disconnected from replication using the `stopReplProducer` failpoint. They remain reachable and can later elect a new primary.
2. The client writes to the current primary, and the local data on each node is inspected.
3. The current primary is killed before the write can replicate to the secondaries.
4. Replication is restored on the secondaries, allowing them to elect a new primary.
5. The client reads and writes through the replica-set connection after failover.
6. The old primary is restarted. Any write that existed only on the old primary must be rolled back before it can rejoin the replica set.

A temporary `stall_marker` write is issued after replication is disconnected. It ensures that replication has fully stopped before the test write is performed. This marker uses `w=1` because a majority write cannot succeed when both secondaries are disconnected.

In [9]:
def run_downed_primary(config_name):
    header("Downed primary", config_name)
    students_primary = get_collection("students", config_name, "primary")

    old_primary = current_primary()
    rollback_lines_before = count_rollback_lines(old_primary)
    secondaries = [node for node in NODES if node != old_primary]

    student_id = f"S005-{config_name}"
    query = {"student_id": student_id}
    reset_document("students", "student_id", make_student(
        student_id, "Alan White", academic_history=["B.Sc. Computer Science"]
    ))

    # 1. Cut both secondaries off, so the next write only reaches the primary
    for node in secondaries:
        disconnect_replication(node)
    try:
        marker = client.get_database(DB_NAME, write_concern=WriteConcern(w=1))["stall_marker"]
        marker.insert_one({"at": time.time()})  # absorbs a write that may slip through
        time.sleep(2)

        # 2. Write, then check which nodes have it
        update = {"$push": {"academic_history": "M.Sc. Data Science"}}
        show("Write result:", try_write(students_primary, query, update))
        show("Primary read:", read_field(students_primary, query, "academic_history"))
        for node in NODES:
            show(f"{node} has:", read_node(node, query, "academic_history"))

        # 3. Kill the primary before the write can replicate
        docker("kill", old_primary)
    except ServerSelectionTimeoutError as error:
        print("Experiment stopped")
    finally:
        for node in secondaries:
            reconnect_replication(node)

    new_primary = current_primary()
    show("New primary:", new_primary)

    # NEW: same client and read concern as "Primary read" before the kill
    show("Client read after failover:", read_field(students_primary, query, "academic_history"))

    # NEW: a later write from the same client
    students_primary.update_one(query, {"$push": {"academic_history": "PhD Information Systems"}})
    show("After writing PhD:", read_field(students_primary, query, "academic_history"))

    # 4. Restart the old primary; to rejoin, it must roll back the write
    docker("start", old_primary)

    # CHANGED: a w=3 write can only be confirmed once the old primary has
    # rolled back and caught up with the new primary's history
    catch_up = client.get_database(DB_NAME, write_concern=WriteConcern(w=3, wtimeout=120000))
    catch_up["stall_marker"].insert_one({"at": time.time()})

    show(f"{old_primary} after rejoining:", read_node(old_primary, query, "academic_history"))
    rolled_back = count_rollback_lines(old_primary) > rollback_lines_before
    show(f"{old_primary} rolled back:", "yes" if rolled_back else "no")

### 2.7  Check the Cluster Before Testing

Run the following cell before starting a failure experiment, and again if an experiment is interrupted.

`restore_cluster()` unpauses and starts all nodes, restores replication, waits for a primary to be elected, and displays the replica-set status. It is safe to run even when the cluster is already healthy.


In [10]:
restore_cluster()

Primary: mongo3
mongo1:27017 SECONDARY 2026-09-22 13:52:10
mongo2:27017 SECONDARY 2026-09-22 13:52:10
mongo3:27017 PRIMARY 2026-09-22 13:52:10


## 3. Node Failure Experiments

This section runs the failure scenarios using both `majority` and `w1-local` configurations.

The experiments investigate how write acknowledgement, read visibility, data durability, and client-centric consistency change when replica-set nodes become unavailable or when the primary fails.

The following scenarios are tested:

1. One secondary node is paused while the primary and one secondary remain available.
2. Two secondary nodes are paused, causing the primary to lose its majority.
3. The primary is killed after a write is issued, forcing a new primary election and possible rollback of an unreplicated write.

### 3.1 Experiment 1: One Disconnected Secondary

One secondary node is paused while the primary and the remaining secondary continue operating. Two of the three nodes remain available, so the replica set retains a majority and no failover is expected.

#### Prediction

| Configuration | Expected Write Outcome | Expected Primary Read | Expected Recovery Behaviour |
|---|---|---|---|
| `w="majority"`, `readConcern="majority"` | **Acknowledged.** The primary and the remaining active secondary form a majority, allowing the write to satisfy the majority write concern. | **Graduate.** The primary should return the majority-committed update. | The paused secondary catches up from the replication log after it is unpaused. |
| `w=1`, `readConcern="local"` | **Acknowledged.** The primary confirms the write immediately after storing it. | **Graduate.** The primary returns its locally stored update. | The paused secondary receives the update and catches up after it is unpaused. |

**What to look for:** Whether the write is acknowledged and readable while one secondary is unavailable, whether the original primary remains primary, and whether the paused node catches up after reconnecting.

In [22]:
run_disconnected_secondaries("majority", how_many=1)

=== Disconnected secondaries: 1 | w=majority, readConcern=majority ===
  [docker pause mongo1]
  Write result:                  acknowledged
  Primary read:                  Graduate
  [docker unpause mongo1]
  mongo1 after reconnecting:     Graduate
  mongo2 after reconnecting:     Graduate
  mongo3 after reconnecting:     Graduate


In [24]:
run_disconnected_secondaries("w1-local", how_many=1)

=== Disconnected secondaries: 1 | w=1, readConcern=local ===
  [docker pause mongo1]
  Write result:                  acknowledged
  Primary read:                  Graduate
  [docker unpause mongo1]
  mongo1 after reconnecting:     Graduate
  mongo2 after reconnecting:     Graduate
  mongo3 after reconnecting:     Graduate


#### Interpretation

- **`majority` configuration**

  - **Result:** The write was acknowledged, the primary read returned `Graduate`, and all three nodes returned `Graduate` after the paused secondary reconnected.

  - **Why it happened:** Although one secondary was paused, the primary and the remaining active secondary still formed a majority. The update was therefore replicated to two nodes before MongoDB acknowledged it. After the paused node was unpaused, it caught up and received the same update.

- **`w1-local` configuration**

  - **Result:** The write was acknowledged, the primary read returned `Graduate`, and all three nodes returned `Graduate` after the paused secondary reconnected.

  - **Why it happened:** With `w=1`, MongoDB acknowledged the update as soon as the primary stored it. The remaining active secondary and the paused secondary received the update later through normal replication. The final result matched the `majority` run because the primary remained available and the paused node was restored successfully.

- **Comparison**

  - Both configurations remained available because one primary and one secondary were still running, which preserved a majority of the three-node replica set.

  - The difference is when the write was acknowledged: `majority` waited for the remaining secondary, whereas `w1-local` acknowledged the write immediately on the primary. This difference would matter if the primary failed before the `w1-local` write replicated.

### 3.2 Experiment 2: Two Disconnected Secondaries

Both secondary nodes are paused, leaving only the original primary available. The primary no longer has a majority of the three-node replica set, so a majority write cannot be confirmed.

#### Prediction

| Configuration | Expected Write Outcome | Expected Primary Read | Expected Recovery Behaviour |
|---|---|---|---|
| `w="majority"`, `readConcern="majority"` | **Not confirmed.** The primary may apply the update locally, but it cannot receive acknowledgement from another node, so the majority write should time out. | **Baseline value.** A majority read should return the earlier majority-committed value, not the unconfirmed update. | After both secondaries are unpaused, the locally applied update should replicate and all nodes should eventually converge. |
| `w=1`, `readConcern="local"` | **Acknowledged.** The primary confirms the update immediately because `w=1` does not require another node. | **Graduate.** The local read on the primary returns the value it stored. | After both secondaries are unpaused, the update should replicate and all nodes should eventually converge. |

**What to look for:** Whether each configuration can confirm the write without any active secondary, what the primary returns immediately after the write attempt, and whether all nodes converge after the paused secondaries reconnect.

**Note:** If both secondaries remain paused for long enough, the primary loses its majority and steps down. No new primary can be elected while both secondaries are unavailable, so the experiment should be completed quickly.

In [25]:
run_disconnected_secondaries("majority", how_many=2)

=== Disconnected secondaries: 2 | w=majority, readConcern=majority ===
  [docker pause mongo1]
  [docker pause mongo3]
  Write result:                  NOT confirmed within 3000 ms
  Primary read:                  Undergraduate
  [docker unpause mongo1]
  [docker unpause mongo3]
  mongo1 after reconnecting:     Graduate
  mongo2 after reconnecting:     Graduate
  mongo3 after reconnecting:     Graduate


In [26]:
run_disconnected_secondaries("w1-local", how_many=2)

=== Disconnected secondaries: 2 | w=1, readConcern=local ===
  [docker pause mongo1]
  [docker pause mongo3]
  Write result:                  acknowledged
  Primary read:                  Graduate
  [docker unpause mongo1]
  [docker unpause mongo3]
  mongo1 after reconnecting:     Graduate
  mongo2 after reconnecting:     Graduate
  mongo3 after reconnecting:     Graduate


#### Interpretation

- **`majority` configuration**

  - **Result:** The write was not confirmed within the timeout, and the primary read returned the earlier value, `Undergraduate`. After the two paused secondaries reconnected, all three nodes returned `Graduate`.

  - **Why it happened:** With both secondaries paused, the primary was the only available node and could not form a majority. It applied `Graduate` locally, but could not confirm the write because no secondary could acknowledge it. The majority read returned `Undergraduate`, the most recent value committed before the secondaries were paused. Once they reconnected, the locally applied update replicated to both nodes.

- **`w1-local` configuration**

  - **Result:** The write was acknowledged, the primary read returned `Graduate`, and all three nodes returned `Graduate` after the secondaries reconnected.

  - **Why it happened:** With `w=1`, MongoDB confirmed the update as soon as the primary stored it, without waiting for either paused secondary. With `readConcern="local"`, the primary immediately returned its own locally stored value, `Graduate`. When the secondaries reconnected, they received the update through replication.

- **Comparison**

  - With both secondaries unavailable, the replica set no longer had a majority. Therefore, the `majority` write could not be confirmed, while the `w1-local` write was acknowledged by the primary alone.

  - Both configurations eventually converged after the secondary nodes reconnected. However, the `w1-local` write was temporarily stored only on the primary and would have been at risk of loss if that primary had failed before replication resumed.

### 3.3 Experiment 3: Primary Failure and Failover

Replication is stopped on both secondaries, so the M.Sc. write reaches only the current primary. The primary is then killed, and the secondaries reconnect to elect a new primary without that write. When the old primary restarts, it rolls back the unreplicated M.Sc. write before rejoining.

**Check the run is valid first:** Before the primary is killed, only the original primary should contain M.Sc. Both secondaries should contain only the baseline history.

#### Prediction

| Configuration | Expected Result Before Primary Failure | Expected Result After Failover | Expected Recovery Behaviour |
|---|---|---|---|
| `w="majority"`, `readConcern="majority"` | **Write not confirmed.** The write cannot reach a majority, and a majority read returns the committed baseline data. | **Baseline remains visible.** The new primary does not contain the unconfirmed write. | The old primary rolls back its unconfirmed write when it rejoins. No client-centric guarantee is violated because the client was never told the write succeeded. |
| `w=1`, `readConcern="local"` | **Write acknowledged.** The old primary stores and returns the update locally, although no secondary has received it. | **Acknowledged write disappears.** A secondary without the update becomes the new primary, so the client no longer sees its earlier acknowledged write. | The old primary rolls back the unreplicated write when it rejoins. This demonstrates possible read-your-writes, monotonic-reads, and monotonic-writes violations after failover. |

**What to look for:** Whether the initial write was acknowledged, what the client reads from the new primary after failover, and whether the old primary rolls back the write after it rejoins.

In [28]:
run_downed_primary("majority")

=== Downed primary | w=majority, readConcern=majority ===
  Write result:                  NOT confirmed within 3000 ms
  Primary read:                  ['B.Sc. Computer Science']
  mongo1 has:                    ['B.Sc. Computer Science']
  mongo2 has:                    ['B.Sc. Computer Science', 'M.Sc. Data Science']
  mongo3 has:                    ['B.Sc. Computer Science']
  [docker kill mongo2]
  New primary:                   mongo3
  Client read after failover:    ['B.Sc. Computer Science']
  After writing PhD:             ['B.Sc. Computer Science', 'PhD Information Systems']
  [docker start mongo2]
  mongo2 after rejoining:        ['B.Sc. Computer Science', 'PhD Information Systems']
  mongo2 rolled back:            yes


In [18]:
show_members()

mongo1:27017 (not reachable/healthy) 1970-01-01 00:00:00
mongo2:27017 SECONDARY 2026-09-22 15:06:41
mongo3:27017 PRIMARY 2026-09-22 15:06:41


In [19]:
run_downed_primary("w1-local")

=== Downed primary | w=1, readConcern=local ===
  Write result:                  acknowledged
  Primary read:                  ['B.Sc. Computer Science', 'M.Sc. Data Science']
  mongo1 has:                    ['B.Sc. Computer Science']
  mongo2 has:                    ['B.Sc. Computer Science']
  mongo3 has:                    ['B.Sc. Computer Science', 'M.Sc. Data Science']
  [docker kill mongo3]
  New primary:                   mongo2
  Client read after failover:    ['B.Sc. Computer Science']
  After writing PhD:             ['B.Sc. Computer Science', 'PhD Information Systems']
  [docker start mongo3]
  mongo3 after rejoining:        ['B.Sc. Computer Science', 'PhD Information Systems']
  mongo3 rolled back:            yes


In [20]:
show_members()

mongo1:27017 SECONDARY 2026-09-22 15:10:50
mongo2:27017 PRIMARY 2026-09-22 15:10:50
mongo3:27017 (not reachable/healthy) 1970-01-01 00:00:00


#### Interpretation

- **`majority` configuration**

  - **Result:** The M.Sc. write was not confirmed within the timeout, and the majority read returned only the baseline history. Although the old primary contained M.Sc. locally, neither secondary contained it. After failover, the new primary returned the baseline history, and the old primary rolled back M.Sc. when it rejoined.

  - **Why it happened:** With `w="majority"`, MongoDB required the write to reach the old primary and at least one secondary before acknowledging it. Since replication to both secondaries was stopped, the write could not satisfy this requirement. The old primary applied M.Sc. locally, but the client was not told the write succeeded and the majority read did not expose it. After the old primary failed, the new primary correctly used the baseline state, and the unconfirmed M.Sc. write was rolled back.

- **`w1-local` configuration**

  - **Result:** The old primary acknowledged the M.Sc. write and returned it through a local read. The two secondaries did not contain M.Sc. After the old primary was killed, the new primary returned only the baseline history. The old primary rolled back M.Sc. when it rejoined.

  - **Why it happened:** With `w=1`, MongoDB acknowledged the M.Sc. write as soon as the old primary stored it. Replication to both secondaries had been intentionally stopped, so neither secondary received the write before failover. A secondary without M.Sc. became the new primary, causing the client’s earlier acknowledged write to disappear. This is a read-your-writes violation. The later PhD write was issued through the new primary, so the final replicated history contained the baseline qualification and PhD, but not M.Sc.

- **Comparison**

  - Both old primaries rolled back M.Sc. because the write had not reached either secondary before the failover.

  - The key difference is acknowledgement: `w1-local` acknowledged a write that later disappeared, while `majority` did not confirm the write. Therefore, only the `w1-local` run violated the client’s expectation that an acknowledged write would remain visible after failover.

## Restoring the Cluster for Normal Operation

The primary node may have changed during the failover experiments. Before returning to the replication-delay experiments in the main notebook, restore the cluster and confirm that `mongo3` is a secondary.

If `mongo3` is the current primary, step it down so that it does not disrupt experiments designed to pause or inspect `mongo3`.

In [27]:
restore_cluster()
# If mongo3 is primary, uncomment:
# direct["mongo3"].admin.command("replSetStepDown", 60)

Primary: mongo2
mongo1:27017 SECONDARY 2026-09-22 15:19:00
mongo2:27017 PRIMARY 2026-09-22 15:19:10
mongo3:27017 SECONDARY 2026-09-22 15:19:10
